In [0]:
from pyspark.sql import functions as F
from datetime import datetime

RAW_ROOT = "/Workspace/Users/mahabubbasha1203@gmail.com/fhir-medallion-lakehouse/data/raw"
BRONZE_DB = "bronze"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")

def raw_to_bronze(resource_name, extraction_date):
    raw_path = f"file:{RAW_ROOT}/{extraction_date}/{resource_name}"
    df = spark.read.option("multiLine", True).json(raw_path)

    exploded = (
        df.select(
            "resource_type",
            "extraction_timestamp",
            "api_url",
            F.explode("response.entry").alias("entry"),
        )
        .select(
            "resource_type",
            "extraction_timestamp",
            F.col("api_url").alias("api_url_or_params"),   # rename to match required metadata column name
            F.col("entry.resource").alias("resource_json"),
            F.col("entry.resource.id").alias("resource_id"),
        )
        .withColumn("bronze_load_date", F.lit(extraction_date))
        .withColumn("ingestion_ts", F.current_timestamp())
    )

    target_table = f"{BRONZE_DB}.{resource_name.lower()}"
    (
        exploded.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )
    print(f"Wrote {exploded.count()} rows to {target_table}")

extraction_date = datetime.utcnow().strftime("%Y-%m-%d")

for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    raw_to_bronze(resource, extraction_date)

In [0]:
%sql
-- Verify all 4 tables are registered in bronze schema
SHOW TABLES IN bronze;

-- Preview patient table
SELECT resource_id, resource_type, extraction_timestamp, api_url_or_params, bronze_load_date
FROM bronze.patient
LIMIT 5;